# Predicting Bridge Performance under Earthquakes - Demo

UE24CS352A Machine Learning Mini-Project, PES University
Nihar S Oak (PES1UG24CS299), Nikita Mankani (PES1UG24CS300)

This notebook just runs the code from our repository (`src/` and `run_experiments.py`) and shows the outputs.
Run the cells from top to bottom. The first cell asks for the project zip (`bridge-eq-ml.zip`).

### Setup

In [ ]:
# upload the project zip and move into the project folder
from google.colab import files
import zipfile, glob, os

uploaded = files.upload()
for name in uploaded:
    zipfile.ZipFile(name).extractall("/content/project")

src_file = [p for p in glob.glob("/content/project/**/src/data_loader.py", recursive=True) if "__MACOSX" not in p][0]
os.chdir(os.path.dirname(os.path.dirname(src_file)))
print("Working folder:", os.getcwd())
print(os.listdir())

In [ ]:
!pip install -q -r requirements.txt

import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

## 1. The dataset

Dataset from Jiang & Zhang (Stanford CS229, 2018): bridges in U.S. and Japanese earthquakes.
Features: material, age, bridge type, earthquake magnitude and distance to the epicenter.
Target: whether the bridge was damaged (0/1). The cleaning is in `src/data_loader.py`.

In [ ]:
from src.data_loader import load_raw, clean

raw = load_raw()
df = clean(raw)

print("Raw file:", raw.shape[0], "rows,", raw.shape[1], "columns")
print("After cleaning:", len(df), "rows")
print("Damaged:", df.damaged.sum(), "| Not damaged:", (df.damaged == 0).sum())
print("Number of bridges:", df.bridge_id.nunique())

df[["bridge_id", "earthquake", "material", "age", "bridge_type", "magnitude", "distance_mi", "damaged"]].head(10)

## 2. A quick look at the data

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

counts = df.damaged.value_counts().sort_index()
ax[0].bar(["Not damaged", "Damaged"], counts.values, color=["steelblue", "crimson"])
for i, v in enumerate(counts.values):
    ax[0].text(i, v + 2, str(v), ha="center")
ax[0].set_title("Class balance")

for label, color, name in [(0, "steelblue", "Not damaged"), (1, "crimson", "Damaged")]:
    part = df[df.damaged == label]
    ax[1].scatter(part.magnitude, part.distance_mi, c=color, label=name, alpha=0.6, edgecolor="k")
ax[1].set_xlabel("Magnitude")
ax[1].set_ylabel("Distance to epicenter (miles)")
ax[1].set_title("Magnitude vs distance")
ax[1].legend()

plt.tight_layout()
plt.show()

## 3. Training and evaluating the models

`run_experiments.py` trains and tunes all the models and saves the results.
Our main evaluation is 5-fold cross-validation where all records of a bridge stay in the same fold,
so the model is always tested on bridges it has not seen during training.

`REPEATS = 5` gives the numbers in our report (takes around 10 minutes on Colab).
`REPEATS = 1` is quicker (around 2-3 minutes) but the numbers change slightly.

In [ ]:
REPEATS = 5
!python run_experiments.py --repeats {REPEATS}

## 4. Results (cross-validation grouped by bridge)

In [ ]:
res = pd.read_csv("results/metrics/C_grouped_cv_binary.csv", index_col=0)
table = res[["accuracy_mean", "precision_mean", "recall_mean", "f1_mean", "roc_auc_mean", "pr_auc_mean"]]
table.columns = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC", "PR-AUC"]
table.index.name = "Model"
table.round(3)

In [ ]:
display(Image("results/figures/C_confusion_matrices.png", width=1000))

In [ ]:
display(Image("results/figures/C_roc_curves.png", width=550))
display(Image("results/figures/C_pr_curves.png", width=550))

## 5. Random split vs grouped split

A: one random 70/30 split (the same bridge can be in train and test).
B: 5-fold CV on rows. C: 5-fold CV grouped by bridge (our main result).

In [ ]:
comp = pd.read_csv("results/metrics/protocol_comparison.csv", index_col=0)
comp = comp[["A_split_F1", "B_rowCV_F1", "C_groupCV_F1"]]
comp.columns = ["F1 - A: 70/30 split", "F1 - B: row CV", "F1 - C: grouped CV"]
comp.index.name = "Model"
comp.round(3)

## 6. Which features matter? (permutation importance)

In [ ]:
display(Image("results/figures/C_permutation_importance.png", width=1100))

## 7. Predicting the damage level (3 classes)

In [ ]:
mc = pd.read_csv("results/metrics/C_grouped_multiclass_mean.csv", index_col=0)
display(mc.loc[["I: No damage", "II: Mod./severe", "III: Collapse"], ["precision", "recall", "f1", "support"]].round(3))
display(Image("results/figures/multiclass_confusion.png", width=900))

## 8. Using the final model

The best model is retrained on all the data and saved in `results/models/final_model.joblib`.
Change the values below to try a different bridge / earthquake.

In [ ]:
import joblib, json

model = joblib.load("results/models/final_model.joblib")
print("Final model:", json.load(open("results/summary.json"))["final_model"])

def predict(material, age, bridge_type, magnitude, distance_mi):
    x = pd.DataFrame([{"material": material, "age": age, "bridge_type": bridge_type,
                       "magnitude": magnitude, "distance_mi": distance_mi}])
    return "Damaged" if model.predict(x)[0] == 1 else "Not damaged"

# material: steel / concrete
# bridge_type: beam_girder / truss / cable_suspension / arch
print("Concrete beam bridge, 30 yrs, M6.7 at 10 miles :", predict("concrete", 30, "beam_girder", 6.7, 10))
print("Concrete beam bridge, 30 yrs, M5.0 at 60 miles :", predict("concrete", 30, "beam_girder", 5.0, 60))
print("Steel truss bridge,   50 yrs, M7.2 at 25 miles :", predict("steel", 50, "truss", 7.2, 25))